In [47]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data = fetch_california_housing()

X = data.data
y = data.target

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42, test_size=0.2)

We’ll intentionally make neural networks fail so you know when NOT to use them.

### Failure Pattern 1 - Very Small Dataset

Setup (too little data)

In [48]:
# only 50 samples
X_small = X_train[:50]
y_small = y_train[:50]

X_small_t = torch.tensor(X_small, dtype=torch.float32)
y_small_t = torch.tensor(y_small, dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

small_ds = TensorDataset(X_small_t, y_small_t)
small_loader = DataLoader(small_ds, batch_size=16, shuffle=True)

Train the same neural network

In [49]:
model = nn.Sequential(
    nn.Linear(X_small.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(30):
    for xb, yb in small_loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

print(loss)

tensor(5.2825, grad_fn=<MseLossBackward0>)


Evaluate on validation set

In [50]:
with torch.no_grad():
    preds = model(X_val_t)
    loss = loss_fn(preds, y_val_t)
print(loss)

tensor(5.8183)


What you’ll see:
- training loss ↓
- validation loss ↑ or terrible

✅ Overfitting

### Failure Pattern 2 - Problem Is Basically Linear

simpler nn:

In [51]:
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 1)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    for xb, yb in small_loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

print(loss)

with torch.no_grad():
    preds = model(X_val_t)
    loss = loss_fn(preds, y_val_t)
print(loss)

tensor(22484.4160, grad_fn=<MseLossBackward0>)
tensor(29430.0527)


You’ll notice:
- performance ≈ Linear Regression
- sometimes worse

Why?
- NN adds no value
- extra complexity hurts

Rule:

> If linear regression works well, don’t use NN.

### Failure Pattern 3 - Too Much Depth for Tabular Data

In [52]:
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 1)
)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    for xb, yb in small_loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

print(loss)

with torch.no_grad():
    preds = model(X_val_t)
    loss = loss_fn(preds, y_val_t)
print(loss)

tensor(6.3278, grad_fn=<MseLossBackward0>)
tensor(1.7940)


What happens:
- training slow
- unstable
- overfitting

Tabular data rarely needs deep networks.

### Failure Pattern 4 - Skipping Baselines

If you:
- jump straight to NN
- don’t test simpler models

You may:
- waste time
- get worse results
- deploy unstable systems

Baseline first. Always.

### Professional decision table

| Situation            | Best choice          |
| -------------------- | -------------------- |
| Small data           | Linear / Tree models |
| Mostly linear        | Linear Regression    |
| Tabular, medium data | Shallow NN           |
| Images               | CNN                  |
| Sequential decisions | RL                   |
